In [1]:
# ============================================
# ICU AI SYSTEM — MORTALITY MODEL
# ============================================

# Core
import pandas as pd
import numpy as np

# Train/Test Split
from sklearn.model_selection import train_test_split

# Scaling
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# XGBoost
from xgboost import XGBClassifier

# Model Saving
import joblib

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================
# LOAD MORTALITY DATASET
# ============================================

df = pd.read_csv(
    "../data/processed/mortality_processed.csv"
)

print("Dataset Loaded")

print("Shape:", df.shape)

df.head()

Dataset Loaded
Shape: (103437, 48)


,Age,Temperature,MeanArterialPressure,HeartRate,RespiratoryRate,FiO2,pO2,pCO2,ArterialpH,Sodium,...,AtmosphericPressure,SystemValue,DiagnosisValue,Gender,ApacheivScore,ApsScore,EstimatedMortalityRate,EstimatedLengthOfStay,APACHE_WARD,Mortality_Target
0,83,36.9,103.00,100.0,20.0,40.0,97.7,43.0,7.42,146.0,...,760.0,5,32,1,43,26,37.0,3.7,135,0
1,44,36.0,84.00,78.0,20.0,30.0,87.0,35.0,7.38,136.0,...,760.0,8,71,0,28,17,6.4,2.0,339,0
2,75,36.7,73.26,104.0,20.0,30.0,141.0,26.0,7.45,138.0,...,760.0,9,94,0,77,60,36.2,6.1,471,0
3,43,37.0,77.00,110.0,16.0,40.0,196.0,34.1,7.40,146.0,...,760.0,1,61,0,53,53,2.6,3.8,229,1
4,51,36.0,98.00,96.0,20.0,30.0,178.0,36.0,7.32,131.0,...,760.0,6,58,0,31,26,10.5,3.2,273,0


In [3]:
# ============================================
# FEATURES AND TARGET
# ============================================

X = df.drop(
    columns=["Mortality_Target"]
)

y = df["Mortality_Target"]

print("Feature Shape:", X.shape)

print("Target Shape:", y.shape)

Feature Shape: (103437, 47)
Target Shape: (103437,)


In [4]:
# ============================================
# SAVE FEATURE ORDER
# ============================================

feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/mortality/mortality_features.pkl"
)

print("Feature Columns Saved")

print(feature_columns)

Feature Columns Saved
['Age', 'Temperature', 'MeanArterialPressure', 'HeartRate', 'RespiratoryRate', 'FiO2', 'pO2', 'pCO2', 'ArterialpH', 'Sodium', 'UrineOutput', 'Creatinine', 'Urea', 'BSL', 'Albumin', 'Bilirubin', 'Hematocrit', 'WBC', 'IsGCSNotAvailable', 'GCSEyes', 'GCSVerbal', 'GCSMotor', 'MecanicalVentilation', 'CRF', 'Lymphoma', 'Cirrhosis', 'Leukemia', 'HepaticFailure', 'Immunosuppression', 'MetastaticCarcinoma', 'AIDS', 'PreICULengthOfStay', 'DiagnosisType', 'Origin', 'EmergencySurgery', 'Readmission', 'Thrombolysis', 'RespiratoryQuotient', 'AtmosphericPressure', 'SystemValue', 'DiagnosisValue', 'Gender', 'ApacheivScore', 'ApsScore', 'EstimatedMortalityRate', 'EstimatedLengthOfStay', 'APACHE_WARD']


In [5]:
# ============================================
# TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)

print("Testing Shape:", X_test.shape)

Training Shape: (82749, 47)
Testing Shape: (20688, 47)


In [6]:
# ============================================
# FEATURE SCALING
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling Completed")


Scaling Completed


In [7]:
# ============================================
# SAVE SCALER
# ============================================

joblib.dump(
    scaler,
    "../models/mortality/mortality_scaler.pkl"
)

print("Scaler Saved Successfully")

Scaler Saved Successfully


In [8]:
# ============================================
# CLASS IMBALANCE HANDLING
# ============================================

alive_count = (y_train == 0).sum()

dead_count = (y_train == 1).sum()

scale_pos_weight = alive_count / dead_count

print("Alive:", alive_count)

print("Dead:", dead_count)

print("Scale Pos Weight:", scale_pos_weight)

Alive: 73256
Dead: 9493
Scale Pos Weight: 7.716843990308648


In [9]:
# ============================================
# TRAIN XGBOOST MORTALITY MODEL
# ============================================

mortality_model = XGBClassifier(

    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",

    eval_metric="logloss",

    random_state=42
)

mortality_model.fit(
    X_train_scaled,
    y_train
)

print("Mortality Model Training Completed")

Mortality Model Training Completed


In [10]:
# ============================================
# PREDICTIONS
# ============================================

y_pred = mortality_model.predict(
    X_test_scaled
)

y_prob = mortality_model.predict_proba(
    X_test_scaled
)[:, 1]

print("Predictions Generated")

Predictions Generated


In [11]:
# ============================================
# MODEL EVALUATION
# ============================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("ROC-AUC Score:", round(roc_auc, 4))

Accuracy: 80.8 %
ROC-AUC Score: 0.8092


In [12]:
# ============================================
# CLASSIFICATION REPORT
# ============================================

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.94      0.83      0.89     18315
           1       0.32      0.60      0.42      2373

    accuracy                           0.81     20688
   macro avg       0.63      0.72      0.65     20688
weighted avg       0.87      0.81      0.83     20688



In [13]:
# ============================================
# CONFUSION MATRIX
# ============================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[15286  3029]
 [  943  1430]]


In [14]:
# ============================================
# FEATURE IMPORTANCE
# ============================================

feature_importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": mortality_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(15))

                   Feature  Importance
44  EstimatedMortalityRate    0.108358
42           ApacheivScore    0.052208
22    MecanicalVentilation    0.045488
29     MetastaticCarcinoma    0.042121
32           DiagnosisType    0.030008
14                 Albumin    0.026156
26                Leukemia    0.024480
28       Immunosuppression    0.022470
5                     FiO2    0.022425
11              Creatinine    0.022267
16              Hematocrit    0.021999
25               Cirrhosis    0.020350
43                ApsScore    0.019667
10             UrineOutput    0.019619
39             SystemValue    0.019206


In [15]:
# ============================================
# SAVE MORTALITY MODEL
# ============================================

joblib.dump(
    mortality_model,
    "../models/mortality/mortality_xgb_model.pkl"
)

print("Mortality Model Saved Successfully")

Mortality Model Saved Successfully


In [16]:
# ============================================
# THRESHOLD TUNING
# ============================================

threshold = 0.35

y_pred_adjusted = (
    y_prob >= threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_pred_adjusted
    )
)

              precision    recall  f1-score   support

           0       0.96      0.69      0.80     18315
           1       0.24      0.78      0.37      2373

    accuracy                           0.70     20688
   macro avg       0.60      0.73      0.59     20688
weighted avg       0.88      0.70      0.75     20688



In [17]:
cm_adjusted = confusion_matrix(
    y_test,
    y_pred_adjusted
)

print(cm_adjusted)

[[12601  5714]
 [  532  1841]]
